# 🔬 Lab 07: Agentic RAG Workflows & Self-Correcting Pipelines with LangGraph

---

## 🎯 Executive Overview & Production Challenge
Traditional RAG is **passive and linear**:
`Query -> Retrieve -> Prompt -> Generate`
If retrieval pulls irrelevant chunks, naive RAG either **hallucinates** or generates misleading answers.

### 🤖 The Agentic RAG Paradigm (Corrective RAG / Self-RAG):
An autonomous agent architecture actively controls its retrieval loop:
1. **Question Intent Router:** Decides if the question requires Vector Retrieval, Graph Lookup, or Direct Answer.
2. **Retrieval Grader Node:** Checks if the retrieved documents are genuinely relevant.
3. **Query Transformation / Re-Writer:** If chunks are irrelevant, rewrites the query and retries.
4. **Web Search Fallback:** If internal documents do not contain the answer, invokes an external search tool.
5. **Hallucination & Faithfulness Grader:** Ensures the generated answer is strictly grounded in the retrieved facts.

```
                 [ User Query ]
                       │
                       ▼
             [ Intent Router Node ]
             ┌─────────┴─────────┐
             ▼                   ▼
    [ Direct Answer ]    [ Vector Retriever ]
                                 │
                                 ▼
                     [ Retrieval Grader ]
                     ┌───────────┴───────────┐
                     ▼                       ▼
            (Chunks Relevant)       (Chunks Irrelevant)
                     │                       │
                     ▼                       ▼
             [ LLM Generator ]      [ Query Rewriter ] ──> [ Retry / Fallback ]
                     │
                     ▼
          [ Hallucination Grader ] ──> [ Deliver Verified Answer ]
```


In [ ]:
import os
import sys
import tempfile
from typing import List, Dict, Any
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from sentence_transformers import SentenceTransformer
import numpy as np

# Configure sandboxed cache directories for Torch, HuggingFace, and Matplotlib
temp_dir = tempfile.gettempdir()
os.environ.setdefault("MPLCONFIGDIR", os.path.join(temp_dir, "matplotlib"))
os.environ.setdefault("TORCH_HOME", os.path.join(temp_dir, "torch"))
os.environ.setdefault("HF_HOME", os.path.join(temp_dir, "hf"))
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", os.path.join(temp_dir, "torch_inductor"))

# Self-healing sys.path resolution so rag_universal_client is ALWAYS found
for candidate in [
    Path.cwd(),
    Path.cwd() / "RAG_ContxtEngg_Labs",
    Path(__file__).parent if "__file__" in locals() else None,
    Path.cwd() / "UST" / "RAG_ContxtEngg_Labs",
    Path(r"c:\Users\jadid\Desktop\MyWork\Techademy\UST\RAG_ContxtEngg_Labs")
]:
    if candidate and candidate.exists() and (candidate / "rag_universal_client.py").exists():
        if str(candidate.resolve()) not in sys.path:
            sys.path.insert(0, str(candidate.resolve()))
        break
import rag_universal_client as rc

# Define typed Agent State Schema
class AgentState(TypedDict):
    question: str
    documents: List[str]
    relevance_grade: str # "RELEVANT" or "IRRELEVANT"
    generation: str
    retry_count: int
    route: str           # "vectorstore" or "direct"

print("[✓] LangGraph Agent State Schema & Multi-Model Engine initialized.")


---
## Step 1: Ingestion Store for the Agent
We initialize a local document store with operational procedures from our turbomachinery manual.


In [ ]:
docs_db = [
    "Section 1.1: Gas turbine normal operating speed N2 is 14,200 to 14,950 RPM with Level 2 ESD trip at 15,650 RPM.",
    "Section 2.1: Anti-surge control valve modulates to open within 350 milliseconds when Delta S falls below 0.08.",
    "Section 2.2: Tandem dry gas seal primary vent leakage flow limit is 45 Nm3/hr per cartridge; blowdown occurs at 90 Nm3/hr.",
    "Section 3.2: Compressor outer barrel casing hydrostatic pressure testing must be performed every 36 months at 1.5x MAWP.",
    "Section 4.3: Engineering and field maintenance operations require verified local Omanization ratios exceeding 68%."
]

embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedder.encode(docs_db, normalize_embeddings=True)
print(f"[✓] Document knowledge base loaded with {len(docs_db)} operational rules.")


---
## Step 2: Defining LangGraph Pipeline Nodes
### 1. Router Node: Route Query
Determines whether query is technical (needs retrieval) or conversational/general.


In [ ]:
def router_node(state: AgentState) -> AgentState:
    q = state["question"].lower()
    technical_keywords = ["speed", "rpm", "valve", "seal", "leak", "hydrostatic", "pressure", "omanization", "trip"]
    if any(k in q for k in technical_keywords):
        state["route"] = "vectorstore"
    else:
        state["route"] = "direct"
    print(f"  [Router] Query classified as: {state['route'].upper()}")
    return state


### 2. Retriever Node
Pulls top-K candidate chunks using cosine similarity.


In [ ]:
def retriever_node(state: AgentState) -> AgentState:
    q = state["question"]
    q_vec = embedder.encode([q], normalize_embeddings=True)[0]
    scores = np.dot(doc_embeddings, q_vec)
    best_idx = int(np.argmax(scores))
    top_chunk = docs_db[best_idx]
    
    state["documents"] = [top_chunk]
    print(f"  [Retriever] Retrieved chunk (Score: {scores[best_idx]:.4f}):\n    '{top_chunk[:80]}...'")
    return state


### 3. Retrieval Grader Node (Corrective RAG)
Grades whether the retrieved document contains sufficient factual signal to answer the question.


In [ ]:
def grader_node(state: AgentState) -> AgentState:
    """
    Evaluates whether the retrieved document is strictly relevant to the user query.
    Uses Anthropic Claude (with Local Ollama backup) as an LLM judge.
    """
    q = state["question"]
    doc = state["documents"][0] if state["documents"] else ""
    
    prompt = (
        f"You are a strict technical retrieval grader for UST Global (UST) turbomachinery operations.\n"
        f"Question: {q}\n"
        f"Retrieved Context: {doc}\n\n"
        f"Does the retrieved context contain technical information, operational limits, or emergency procedures "
        f"directly relevant to answering the question?\n"
        f"Respond with ONLY the single word 'RELEVANT' or 'IRRELEVANT'. Do not include explanation."
    )
    
    res = rc.call_rag_llm(prompt, temperature=0.0, max_tokens=10)
    ans = res["text"].strip().upper()
    state["relevance_grade"] = "RELEVANT" if "RELEVANT" in ans and "IRRELEVANT" not in ans else "IRRELEVANT"
        
    print(f"  [Grader LLM ({res['tier_used']})] Document evaluated as: {state['relevance_grade']}")
    return state


### 4. Query Re-Writer Node (Self-Correction Loop)
If retrieval was graded IRRELEVANT, reformulates the query and retries up to 2 times.


In [ ]:
def rewriter_node(state: AgentState) -> AgentState:
    """
    Reformulates ambiguous or out-of-domain queries into precise engineering searches.
    Uses Anthropic Claude (with Local Ollama backup).
    """
    state["retry_count"] = state.get("retry_count", 0) + 1
    
    prompt = (
        f"The following query failed to retrieve relevant operational documents: '{state['question']}'.\n"
        f"Rewrite this query into a precise engineering search term for UST gas compression manuals.\n"
        f"Focus on standard turbomachinery components, safety thresholds, or instrumentation.\n"
        f"Return ONLY the rewritten query text. Do not include quotes or conversational preamble."
    )
    
    res = rc.call_rag_llm(prompt, temperature=0.2, max_tokens=50)
    state["question"] = res["text"].strip()
    print(f"  [Rewriter LLM ({res['tier_used']})] Reformulated query (Attempt {state['retry_count']}): '{state['question']}'")
    return state


### 5. Response Generator & Hallucination Guard Node
Synthesizes the final answer using retrieved context or delivers a safe fallback.


In [ ]:
def generator_node(state: AgentState) -> AgentState:
    """
    Synthesizes the final grounded engineering response citing retrieved documentation.
    Uses Anthropic Claude (with Local Ollama backup).
    """
    if state["route"] == "direct":
        prompt = (
            f"You are the UST Global (UST) Operational Turbomachinery Assistant. "
            f"Respond politely and concisely to this greeting or direct query: '{state['question']}'. "
            f"Invite the user to ask about compressor envelopes, dry gas seals, anti-surge valves, or maintenance."
        )
        res = rc.call_rag_llm(prompt, temperature=0.2, max_tokens=100)
        state["generation"] = res["text"].strip()
    elif state["relevance_grade"] == "RELEVANT":
        doc = state["documents"][0]
        prompt = (
            f"You are an enterprise AI assistant for UST Global (UST) turbomachinery operations.\n"
            f"Answer the user query strictly using the verified operational context below.\n\n"
            f"Verified Context:\n{doc}\n\n"
            f"User Question: {state['question']}\n\n"
            f"Provide a clear, authoritative response. Cite the specific Section number and exact numerical thresholds."
        )
        res = rc.call_rag_llm(prompt, temperature=0.1, max_tokens=350)
        state["generation"] = f"[Verified Operational Answer ({res['tier_used']})]:\n{res['text'].strip()}"
    else:
        state["generation"] = (
            "[Safe Guardrail Triggered]: Unable to verify this query against official UST operational manuals "
            "after query reformulation. Bypassing ungrounded generation to prevent operational safety risks."
        )
    
    print(f"  [Generator LLM] Final output generated ({len(state['generation'])} chars).")
    return state


---
## Step 3: Compiling the LangGraph State Graph
We wire the nodes together with conditional branching logic.


In [ ]:
# Create workflow
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("router", router_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("grader", grader_node)
workflow.add_node("rewriter", rewriter_node)
workflow.add_node("generator", generator_node)

# Add edges & conditional transitions
workflow.set_entry_point("router")

def route_decision(state: AgentState):
    return "retriever" if state["route"] == "vectorstore" else "generator"

workflow.add_conditional_edges("router", route_decision, {
    "retriever": "retriever",
    "generator": "generator"
})

workflow.add_edge("retriever", "grader")

def grade_decision(state: AgentState):
    if state["relevance_grade"] == "RELEVANT":
        return "generator"
    elif state.get("retry_count", 0) < 1:
        return "rewriter"
    else:
        return "generator"

workflow.add_conditional_edges("grader", grade_decision, {
    "generator": "generator",
    "rewriter": "rewriter"
})

workflow.add_edge("rewriter", "retriever")
workflow.add_edge("generator", END)

app = workflow.compile()
print("[✓] LangGraph Agentic RAG State Machine compiled successfully!")


---
## Step 4: Executing Agentic Test Scenarios
### Scenario A: Relevant Technical Query


In [ ]:
print("--- TEST SCENARIO A: Relevant Operational Query ---")
res_a = app.invoke({
    "question": "What is the primary vent leakage flow limit for dry gas seals?",
    "documents": [],
    "relevance_grade": "",
    "generation": "",
    "retry_count": 0,
    "route": ""
})
print(f"\nFINAL RESPONSE:\n{res_a['generation']}\n")


### Scenario B: Out-of-Domain / Ambiguous Query (Triggers Safe Rejection)


In [ ]:
print("--- TEST SCENARIO B: Out-of-Domain Query ---")
res_b = app.invoke({
    "question": "What is the best restaurant in Muscat for dinner tonight?",
    "documents": [],
    "relevance_grade": "",
    "generation": "",
    "retry_count": 0,
    "route": ""
})
print(f"\nFINAL RESPONSE:\n{res_b['generation']}\n")


---
## 🏁 Production Engineering Takeaways & Exercises
1. **Never use static linear RAG for enterprise chatbots:** Agentic routing and grading prevent 90% of hallucinations.
2. **Implement retry counters:** Always cap query rewrites (`retry_count < 2`) to prevent infinite execution loops and latency spikes.
